In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("../model/minimind_tokenizer")

/Users/kevinlights/app/miniforge3/envs/minimind/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer

PreTrainedTokenizerFast(name_or_path='../model/minimind_tokenizer', vocab_size=6400, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [4]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from model.LMConfig import LMConfig
from model.model import MiniMindLM
from model.dataset import PretrainDataset
import torch

In [5]:
lm_config = LMConfig(
    dim=64,
    n_layers=8,
    n_heads=8,
    n_kv_heads=2,
    vocab_size=6400,
    hidden_dim=None,
    multiple_of=64,
    norm_eps=0.00001,
    max_seq_len=512,
    rope_theta=1000000,
    dropout=0,
    flash_attn=True,
    use_moe=False,
    num_experts_per_tok=2,
    n_routed_experts=4,
    n_shared_experts=True,
    scoring_func="softmax",
    aux_loss_alpha=0.1,
    seq_aux=True,
    norm_topk_prob=True,
)



model = MiniMindLM(lm_config).to("mps")
# model = MiniMindLM(lm_config).to("mps", dtype=torch.float32)
print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad)}")
print(512*6400*2 + (512*512*2 + 512*128*2 + 512*1408*3)*8)
print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.3}M")
model

787520
29097984
0.788M


MiniMindLM(
  (tok_embeddings): Embedding(6400, 64)
  (dropout): Dropout(p=0, inplace=False)
  (layers): ModuleList(
    (0-7): 8 x MiniMindBlock(
      (attention): Attention(
        (wq): Linear(in_features=64, out_features=64, bias=False)
        (wk): Linear(in_features=64, out_features=16, bias=False)
        (wv): Linear(in_features=64, out_features=16, bias=False)
        (wo): Linear(in_features=64, out_features=64, bias=False)
        (attn_dropout): Dropout(p=0, inplace=False)
        (resid_dropout): Dropout(p=0, inplace=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
      (feed_forward): FeedForward(
        (w1): Linear(in_features=64, out_features=192, bias=False)
        (w2): Linear(in_features=192, out_features=64, bias=False)
        (w3): Linear(in_features=64, out_features=192, bias=False)
        (dropout): Dropout(p=0, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=64, out_features=6400, bias=Fals

In [5]:
tokenizer.pad_token_id

0

In [6]:
with open("../dataset/pretrain_hq_mini.jsonl", "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        print(f'line: {line}')
        import json
        sample = json.loads(line.strip())
        print(f'sample: {sample}')
        text = f"{tokenizer.bos_token}{str(sample['text'])}{tokenizer.eos_token}"
        print(f"text: {text}")
        encoding = tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        print(encoding.input_ids.squeeze())
        break

line: {"text": "<s>请从文章中提取出垃圾分类的好处。垃圾分类的好处包括：可以减少垃圾的产生，使得可回收物得到有效利用，减轻垃圾对环境的污染；可以提高生活质量，使我们的生存环境更干净、更美好；可以培养人们良好的生活习惯。</s> <s>请给我介绍一部最近热播的电影。 近期上映的热门电影有《哆啦A梦：伴我同行2》、《寻梦环游记》、《唐人街探案3》等。这些电影不同的题材，都拥有着惊人的制作和精彩的剧情，相信您一定能找到自己喜欢的电影。</s> <s>从这篇文章中提取出减肥动作。在这篇文章中，介绍了许多适用于减肥的瑜伽动作，这些动作包括：\n1.双脚平放式\n2.平衡式\n3.弓式\n这些动作可以锻炼核心肌群和腿部肌肉，消耗更多的热量，从而实现减肥目标。</s> <s>我该如何克服拖延症？您可以尝试制定清晰、可行的计划来完成任务，紧盯具体的时间表，把目标分解成更小的任务，以创造小成功来鼓舞自己。同时，学习适当的时间管理技巧，如使用专注定时器、打破大任务等等的时间。</s>"}

sample: {'text': '<s>请从文章中提取出垃圾分类的好处。垃圾分类的好处包括：可以减少垃圾的产生，使得可回收物得到有效利用，减轻垃圾对环境的污染；可以提高生活质量，使我们的生存环境更干净、更美好；可以培养人们良好的生活习惯。</s> <s>请给我介绍一部最近热播的电影。 近期上映的热门电影有《哆啦A梦：伴我同行2》、《寻梦环游记》、《唐人街探案3》等。这些电影不同的题材，都拥有着惊人的制作和精彩的剧情，相信您一定能找到自己喜欢的电影。</s> <s>从这篇文章中提取出减肥动作。在这篇文章中，介绍了许多适用于减肥的瑜伽动作，这些动作包括：\n1.双脚平放式\n2.平衡式\n3.弓式\n这些动作可以锻炼核心肌群和腿部肌肉，消耗更多的热量，从而实现减肥目标。</s> <s>我该如何克服拖延症？您可以尝试制定清晰、可行的计划来完成任务，紧盯具体的时间表，把目标分解成更小的任务，以创造小成功来鼓舞自己。同时，学习适当的时间管理技巧，如使用专注定时器、打破大任务等等的时间。</s>'}
text: <s><s>请从文章中提取出垃圾分类的好处。垃圾分类的好处包括：可以减少垃圾的产生，使得可回收物得到有效利用，减轻垃圾对环境的污染；可以提高生活质量，使我们的生存环境更干净、更美好；

In [7]:
encoding?

Type:        BatchEncoding
String form:
{'input_ids': tensor([[   1,    1, 1055, 1083, 2239,  413, 5775,  557, 4489, 2467,  269,  587,
           <...> , 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
           0, 0, 0, 0, 0, 0, 0, 0]])}
Length:      3
File:        ~/app/miniforge3/envs/minimind/lib/python3.10/site-packages/transformers/tokenization_utils_base.py
Docstring:  
Holds the output of the [`~tokenization_utils_base.PreTrainedTokenizerBase.__call__`],
[`~tokenization_utils_base.PreTrainedTokenizerBase.encode_plus`] and
[`~tokenization_utils_base.PreTrainedTokenizerBase.batch_encode_plus`] methods (tokens, attention_masks, etc).

This class is derived from a python dictionary and can be used as a dictionary. In addition, this class exposes
utility methods to map from word/character space to token space.

Args:
    data (`dict`, *optional*):
        Dictionary of lists/arrays/tensors returned by the `__call__`/`encode_plus`/`batch_encode_plus` methods
        ('in

In [8]:
# train_ds = PretrainDataset("../dataset/pretrain_hq_mini.jsonl", tokenizer, max_length=512)
train_ds = PretrainDataset("../dataset/pretrain_hq.jsonl", tokenizer, max_length=512)
for X, Y, loss_mask in train_ds:
    print(X, Y, loss_mask)
    break

tensor([   1,    1,  168,  234,  115, 1415,  333, 1335,  413,  677, 2239,  269,
        2397,  315, 2669,  270, 1373, 4819,  496,  341, 1413,  965,  341,  677,
        1486,  508,  286,  590,  991, 1355, 1027, 2239, 3041, 3720,  168,  234,
         115, 1415, 2086, 2397,  315, 2669,  286,    2,  223,    1, 1188,  270,
        2525, 1086,  397, 1698, 2336, 4668,  269, 3858, 5027, 1355,   33, 4668,
         269, 3858, 3897,  724, 4222,  763, 2922,  286, 4600, 5156,  397, 1086,
         608, 1698, 3445, 2404,  371, 4222,  269, 3858, 3274,  814,    2,  223,
           1, 1970,  953,  457,  120, 2941, 2121,  270,  627,  541,  909,  954,
        4172,  568,  265,  228,  829,  269,  457,  120, 2941,  286, 1188,  270,
         397, 2434, 1086,  803, 1970,  953,  457,  120, 2941, 2121,  270,  457,
         120, 2941,  658,  368,  909,  954, 4172,  568,  265,  228,  829, 1552,
         507, 1211, 1122,  286,    2,  223,    1,  428, 1104, 3713, 1282,  333,
        5565, 2208,  355,  541, 5962, 20

/Users/kevinlights/projects/llm-research/minimind/research/../model/dataset.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(input_ids[:-1], dtype=torch.long)
/Users/kevinlights/projects/llm-research/minimind/research/../model/dataset.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y = torch.tensor(input_ids[1:], dtype=torch.long)
/Users/kevinlights/projects/llm-research/minimind/research/../model/dataset.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_mask = torch.tensor(loss_mask[1:], dtype=torch.long

In [9]:
from torch.utils.data.dataloader import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    pin_memory=True,
    drop_last=False,
    shuffle=False,
    num_workers=1,
    sampler=None
)

In [10]:
iter_per_epoch = len(train_loader)
print(iter_per_epoch, len(train_ds)/8)

176638 176637.875


In [11]:
import torch 
torch.is_autocast_enabled(), torch.get_autocast_gpu_dtype()

(False, torch.float16)

In [13]:
from torch import nn, optim
import torch.distributed as dist
import math
import time
import torch 
from contextlib import nullcontext
import warnings

lr = 1e-4
total_epochs = 1
accumulation_steps = 8
grad_clip = 1.0
log_interval = 200
save_interval = 200
save_dir = "output"
optimizer = optim.AdamW(model.parameters(), lr=lr)
ctx = nullcontext()
device = "mps"

warnings.filterwarnings('ignore')

os.makedirs(save_dir, exist_ok=True)

def train():
    for epoch in range(total_epochs):
        start_time = time.time()
        loss_func = nn.CrossEntropyLoss(reduction='none')
        for step, (X, Y, loss_mask) in enumerate(train_loader):
            X = X.to(device)
            Y = Y.to(device)
            loss_mask = loss_mask.to(device)
            # X = X.to(device, dtype=torch.float32)
            # Y = Y.to(device, dtype=torch.float32)
            # loss_mask = loss_mask.to(device, dtype=torch.float32)

            current_step = epoch * iter_per_epoch + step
            total_steps = total_epochs * iter_per_epoch
            
            # 余弦退火学习率动态调整，最低学习率 lr/10，右边公式让学习率按照余弦曲线从初始值下降到最低值
            # current_step/total_steps 从 0 逐渐接近 1，余弦值从 1 降到 -1，整个表达式就从 lr 慢慢降到 lr/10
            # 动态调整学习率的好处是：
            # 开始阶段学习率下降较慢，有利于模型在初期快速学习 
            # 后期学习率缓慢接近最小值，有利于模型精细调整
            # 避免了学习率突变对训练造成的影响
            # 相比单纯线性下载更符合训练过程的实际需求
            lr = lr / 10 + 0.5 * lr * (1 + math.cos(math.pi * current_step / total_steps))

            # 有些模型不同部分可能使用不同的学习率，所有会有多个参数组
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr

            with ctx:
                # res 通常包含 logits: 模型的原始预测值，尚未经过 softmax 归一化
                # aux_loss: 辅助损失，某些模型会有额外的正则化损失
                res = model(X)
                loss = loss_func(
                    res.logits.view(-1, res.logits.size(-1)), # 调整 logits 形状，把 logits 从 [batch_size, seq_len, vocab_size] 变成 [batch_size * seq_len, vocab_size]，这是因为交叉熵损失通常要求 logits 是二维的
                    Y.view(-1) # 调整标签形状，从 [batch_size, seq_len] 变成 [batch_size * seq_len]
                ).view(Y.size()) # 恢复 loss 形状，恢复成 [batch_size, seq_len]
                # 对损失值进行加权平均，只计算有效部分，忽略 padding 等无效位置
                loss = (loss * loss_mask).sum() / loss_mask.sum()
                # 加上模型的辅助扣件
                loss += res.aux_loss
                # 按梯度累积步数缩放损失值，相当于把多个小 batch 的损失平均分摊到每一步，这样在 loss.backward() 时，梯度会被自动累加，但当前损失值看起来更小，避免数值过大
                loss = loss / accumulation_steps
            
            loss.backward()

            if (step + 1) % accumulation_steps == 0:
                # 手动梯度裁剪，防止梯度爆炸
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

                # for name, param in model.named_parameters():
                #     if param.grad is not None:
                #         print(f"{name}: max_grad={param.grad.abs().max()}, mean_grad={param.grad.mean()}")
                
                if torch.isnan(loss).any() or torch.isinf(loss).any():
                    print("Loss contains NaN/Inf!")
                else:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            if step % log_interval == 0:
                spend_time = time.time() - start_time
                print(f"epoch: {epoch+1}/{total_epochs} step: {step} epoch_steps: {iter_per_epoch} loss: {loss.item() * accumulation_steps:.3f} lr: {optimizer.param_groups[-1]['lr']} spend_time: {spend_time/60:.1f}m left_time: {iter_per_epoch/(step+1)*spend_time/60-spend_time/60:.1f}m")

            # dist.get_rank() 是获取当前进程在分布式训练环境中的唯一标识，从 0 开始，0 是主进程，负责协调，保存模型等
            # if (step + 1) % save_interval == 0 and dist.get_rank() == 0:
            if (step + 1) % save_interval == 0:
                model.eval()
                ckp = f"{save_dir}/pretrain_{lm_config.dim}.pth"
                state_dict = model.state_dict()
                torch.save(state_dict, ckp)
                model.train()

# train()


In [14]:
tokenizer = AutoTokenizer.from_pretrained("../model/minimind_tokenizer")
ckp = f"{save_dir}/pretrain_{lm_config.dim}.pth"
model = MiniMindLM(
    LMConfig(
        dim=lm_config.dim,
        n_layers=lm_config.n_layers,
        max_seq_len=lm_config.max_seq_len,
        use_moe=lm_config.use_moe,
    )
)
state_dict = torch.load(ckp, map_location=device)
model.load_state_dict({k: v for k, v in state_dict.items() if 'mask' not in k}, strict=True)
model = model.eval().to(device)

In [16]:
messages = [{"role": "user", "content": "你好"}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
with torch.no_grad():
    x = torch.tensor(tokenizer(prompt)["input_ids"], device=device).unsqueeze(0)
    outputs = model.generate(
        x, 
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=512,
        temperature=0.85,
        top_p=0.85,
        stream=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    print('🤖️: ', end='')
    try:
        history_idx = 0
        for y in outputs:
            answer = tokenizer.decode(y[0].tolist(), skip_special_tokens=True)
            if (answer and answer[-1] == '�') or not answer:
                continue
            print(answer[history_idx:], end='', flush=True)
            history_idx = len(answer)
    except StopIteration:
        print("No answer")
    print('\n')
messages.append({"role": "assistant", "content": answer})
print(messages)

🤖️: 

[{'role': 'user', 'content': '你好'}, {'role': 'assistant', 'content': ''}]


In [ ]:
from IPython.display import display, Markdown   

In [ ]:

mermaid_code = "graph TD\n"
for name, layer in model.named_children():
    mermaid_code += f'    {name}[{name}] --> '
print(mermaid_code)
# display(Markdown(mermaid_code))

graph TD
    tok_embeddings[tok_embeddings] -->     dropout[dropout] -->     layers[layers] -->     norm[norm] -->     output[output] --> 


```mermaid
graph TD
    tok_embeddings[tok_embeddings] -->     dropout[dropout] -->     layers[layers] -->     norm[norm] -->     output[output]
```

In [ ]:
import torch
from torch.nn import Parameter

weight = Parameter(torch.randn(3, 4))
print(weight.numel())

12


In [ ]:

# for p in model.parameters():
#     if not p.requires_grad:
#         continue
#     print(p.numel()) # number of elements

print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.3}M")

25.8M


In [ ]:
# lm_config = LMConfig(
#     dim=64,
#     n_layers=8,
#     n_heads=8,
#     n_kv_heads=2,
#     vocab_size=6400,
#     hidden_dim=None,
#     multiple_of=64,
#     norm_eps=0.00001,
#     max_seq_len=512,
#     rope_theta=1000000,
#     dropout=0,
#     flash_attn=True,
#     use_moe=False,
#     num_experts_per_tok=2,
#     n_routed_experts=4,
#     n_shared_experts=True,
#     scoring_func="softmax",
#     aux_loss_alpha=0.1,
#     seq_aux=True,
#     norm_topk_prob=True,
# )

# from transformers.modeling_utils import PreTrainedModel
# from torch import nn
# import torch


# class RMSNorm(nn.Module):
#     def __init__(self, dim: int, eps: float):
#         super().__init__()
#         self.eps = eps
#         self.weight = nn.Parameter(torch.ones(dim))

#     def forward(self, x: torch.Tensor):
#         return self.weight * (
#             x.float() * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
#         ).type_as(x)


# class Attention(nn.Module):
#     def __init__(self, config: LMConfig):
#         super().__init__()

#     def forward(self, x: torch.Tensor):
#         pass


# class MiniMindBlock(nn.Module):
#     def __init__(self, layer_id: int, config: LMConfig):
#         super().__init__()

#         self.attention = Attention(config)
#         self.attention_norm = RMSNorm(dim=config.dim, eps=config.norm_eps)

#     def forward(self, x: torch.Tensor):
#         self.attention(
#             self.attention_norm(x),
#         )


# class MiniMindLM(PreTrainedModel):
#     config_class = LMConfig

#     def __init__(self, config: LMConfig):
#         self.params = config or LMConfig()
#         super().__init__(self.params)

#         self.vocab_size = config.vocab_size
#         self.n_layers = config.n_layers
#         self.tok_embeddings = nn.Embedding(config.vocab_size, config.dim)
#         self.dropout = nn.Dropout(config.dropout)
#         self.layers = nn.ModuleList([l for l in range(self.n_layers)])